

---


# Freesound audio corpus generation with geotags


> This notebook contains code for downloading sounds and geotag collection from freesound.
> The file paths to audio and its corresponding geotag will be saved to `mappings.json`

---




In [13]:
import os
import json
import requests
import numpy as np
import pandas as pd
import freesound
from dotenv import load_dotenv
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

CACHE_DIR = os.path.join('..', '.cache')
os.makedirs(CACHE_DIR, exist_ok=True)

### Freesound API

> Create `.env` file `FREESOUND_API_KEY=your_api_key`

In [14]:
load_dotenv()

FREESOUND_API_KEY = os.getenv('FREESOUND_API_KEY')

freesound_client = freesound.FreesoundClient()
freesound_client.set_token(FREESOUND_API_KEY)

## Download dataset

In [15]:
def save_audio(audio_url, file_name):
    base_directory = os.path.join(CACHE_DIR, "audio")
    os.makedirs(base_directory, exist_ok=True)
    
    # Download sound from freesound
    audio = requests.get(audio_url)

    # Define file name for audio
    file_path = os.path.join(base_directory, file_name)

    # Write audio
    with open(file_path, "wb") as file:
      file.write(audio.content)

In [16]:
def download_corpus(num_samples=100):
    url = "https://freesound.org/apiv2/search/text/"
    params = {
        "filter": "is_geotagged:1%20duration: [1 TO 60]",
        "fields": "geotag,tags,id,previews",
        "page_size": 150,
        "token": FREESOUND_API_KEY,
    }
    
    # Collect all download tasks
    tasks = []
    mappings = dict()
    response = requests.get(url, params=params).json()
    count = 0
    pbar = tqdm(total=num_samples, desc="Fetching API")
    while response.get('next') is not None and count <= num_samples:
        for i in range(len(response['results'])):
            audio_url = response['results'][i]['previews']['preview-hq-mp3']
            sound_id = response['results'][i]['id']
            file_name = f"{sound_id}.mp3"
            geotag = response['results'][i].get('geotag')
            mappings[file_name] = geotag
            tasks.append((audio_url, file_name))
            count += 1
            pbar.update(1)
            if count >= num_samples:
                break
        if count < num_samples and response.get('next'):
            response = requests.get(url=response['next'], params=params).json()
    pbar.close()
    
    pbar = tqdm(total=len(tasks), desc="Downloading")
    with ThreadPoolExecutor(max_workers=16) as executor:
        futures = {executor.submit(save_audio, url, name): name for url, name in tasks}
        for future in as_completed(futures):
            future.result()
            pbar.update(1)
    pbar.close()
    
    with open(os.path.join(CACHE_DIR, 'data.json'), 'w') as f:
        json.dump(mappings, f, indent=4)

In [17]:
download_corpus(5000)

Fetching API:   0%|          | 0/5000 [00:00<?, ?it/s]

Downloading:   0%|          | 0/5001 [00:00<?, ?it/s]